What does it take to compile a list of Homeric hapax legomena—words that appear exactly once across the *Iliad* and the *Odyssey*? With a fully lemmatized treebank in hand, the answer is: a few lines of Python.

This notebook pulls the Ancient Greek and Latin Dependency Treebank (AGLDT) version of Homer from [`PerseusDL/treebank_data`](https://github.com/PerseusDL/treebank_data), iterates over every annotated `<word>`, collects the `lemma` attribute, and counts. The hapaxes fall out of `collections.Counter` for free; the only nontrivial step is sorting them—`sorted()` on Greek strings will not give you the order you want, so we hand off to [`pyuca`](https://pypi.org/project/pyuca/) for proper Unicode collation.

In [ ]:
# Imports
import urllib.request
from collections import Counter
from random import sample

from lxml import etree
from pyuca import Collator

In [ ]:
# pyuca's Collator implements UCA-correct sort keys for Greek
c = Collator()

In [ ]:
# AGLDT URIs for Homer's Iliad (tlg0012.tlg001) and Odyssey (tlg0012.tlg002)
treebank_base = (
    'https://raw.githubusercontent.com/PerseusDL/treebank_data/master/v2.1/Greek/texts/'
)
works = ['tlg0012.tlg001', 'tlg0012.tlg002']
uris = [f'{treebank_base}{w}.perseus-grc1.tb.xml' for w in works]

In [ ]:
def get_words(uri):
    with urllib.request.urlopen(uri) as f:
        tree = etree.parse(f)
    return tree.getroot().xpath('.//word')


words = []
for uri in uris:
    words.extend(get_words(uri))

print(f"There are {len(words)} 'words' in the AGLDT version of Homer's *Iliad* and *Odyssey*.")

In [ ]:
# Get forms and lemmas from word elements

def get_lemma(word):
    return word.attrib.get('lemma')


forms = [word.attrib['form'] for word in words]
lemmas = [get_lemma(word) for word in words]

unique_forms = sorted(set(forms))
unique_lemmas = sorted({lemma for lemma in lemmas if lemma})

print(f'There are {len(unique_forms)} unique forms in the AGLDT Homer.')
print(f'There are {len(unique_lemmas)} unique lemmas in the AGLDT Homer.')

In [ ]:
# Get hapaxes

hapaxes = [lemma for lemma, count in Counter(lemmas).most_common() if count == 1]

print(f'There are {len(hapaxes)} hapaxes in the AGLDT Homer.')
print(f'A sample of AGLDT hapaxes includes:\n {sample(hapaxes, 10)}')

Sorting Greek strings with `sorted()` alone will use code-point order, which puts smooth-breathing forms before rough-breathing ones in ways that no Greek reader expects. `pyuca.Collator` produces sort keys that follow the [Unicode Collation Algorithm](https://www.unicode.org/reports/tr10/), giving the alphabetical order you'd actually want.

In [ ]:
for hapax in sorted(hapaxes, key=c.sort_key)[:50]:
    print(hapax)

In [ ]:
with open('data/homeric_hapaxes.txt', 'w') as f:
    for hapax in sorted(hapaxes, key=c.sort_key):
        f.write(f'{hapax}\n')